# LeetCode 3930 — Power Update After K-th Largest Insertion II bằng Python

- Link: https://leetcode.com/problems/power-update-after-k-th-largest-insertion-ii/
- Độ khó: **Hard**
- Chủ đề: Thống kê thứ tự (order statistics), danh sách có thứ tự (`bisect`), cây Fenwick (BIT) tìm phần tử thứ `k`, nén tọa độ, lũy thừa modulo

> **Bài Premium: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**
> Bài Premium không có skeleton công khai. Tên phương thức `powerUpdate(self, nums, p, queries)` lấy theo lời giải tham khảo của doocs — **cần đối chiếu** lại chữ ký (kiểu trả về, tên tham số) khi nộp.

Skeleton (theo doocs):

```python
class Solution:
    def powerUpdate(self, nums: List[int], p: int, queries: List[List[int]]) -> List[int]:
```

## 1. Giới thiệu & mục tiêu học

Cho mảng `nums`, một số `p` và các truy vấn `[val, k]`. Với mỗi truy vấn theo thứ tự:

1. **Chèn** `val` vào `nums`.
2. Gọi `x` là phần tử lớn thứ `k` trong `nums` hiện tại.
3. Cập nhật `p = p^x mod (10^9 + 7)`.

Trả về `p` sau mỗi truy vấn.

Hai việc con độc lập: **(a)** duy trì một tập số có thứ tự cho phép **chèn** và **hỏi phần tử lớn thứ `k`**; **(b)** tính `p^x mod M` nhanh bằng lũy thừa nhị phân (`pow(p, x, MOD)` của Python). Phần khó và hay là (a) khi `k` thay đổi tùy ý theo mỗi truy vấn nên **heap** không giúp được.

Sau notebook này bạn sẽ:

1. Dùng `bisect.insort` để giữ danh sách luôn được sắp xếp, và hiểu chi phí `O(n)` dịch phần tử của nó.
2. Cài **cây Fenwick** đếm tần suất trên giá trị đã nén và **tìm phần tử thứ `k`** bằng bước nhảy nhị phân `O(log n)`.
3. Cài một **danh sách có thứ tự theo khối** (mini `SortedList`) với truy cập theo hạng.
4. Biết vì sao cập nhật `p` phải dùng `p` **hiện tại** và `pow` ba tham số.
5. Viết 4 lời giải (sort lại mỗi lần, `insort`, Fenwick, danh sách khối) và đối chiếu bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `nums`, `p`, `queries[i] = [val_i, k_i]`. Với từng truy vấn `i` (theo thứ tự): chèn `val_i`, lấy `x` = phần tử lớn thứ `k_i` (`k = 1` là lớn nhất), đặt `p ← p^x mod (10^9 + 7)`, ghi `p` vào đáp án.

**Ví dụ 1:** `nums = [2]`, `p = 4`, `queries = [[3,1],[1,2]]` → `[64, 4096]`.
- Chèn 3 → `[2,3]`, lớn thứ 1 là `3`, `p = 4^3 = 64`.
- Chèn 1 → `[2,3,1]`, lớn thứ 2 là `2`, `p = 64^2 = 4096`.

**Ví dụ 2:** `nums = [7,5]`, `p = 6`, `queries = [[4,3],[7,2]]` → `[1296, 220296870]`.
- Chèn 4 → `[7,5,4]`, lớn thứ 3 là `4`, `p = 6^4 = 1296`.
- Chèn 7 → `[7,5,4,7]`, lớn thứ 2 là `7` (hai số 7 là hạng 1 và 2), `p = 1296^7 mod (10^9+7) = 220296870`.

**Ràng buộc:** `1 ≤ len(nums) ≤ 2·10^4`, `1 ≤ nums[i] ≤ 10^9`, `1 ≤ p ≤ 10^9`, `1 ≤ len(queries) ≤ 2·10^4`, `1 ≤ val ≤ 10^9`, `1 ≤ k_i ≤ n + i + 1` (luôn hợp lệ sau khi chèn).

In [1]:
import random, time, bisect
from typing import List

MOD = 10**9 + 7
EXAMPLES = [
    (([2], 4, [[3, 1], [1, 2]]), [64, 4096]),
    (([7, 5], 6, [[4, 3], [7, 2]]), [1296, 220296870]),
]
for args, exp in EXAMPLES:
    print(args, "-> kỳ vọng", exp)

([2], 4, [[3, 1], [1, 2]]) -> kỳ vọng [64, 4096]
([7, 5], 6, [[4, 3], [7, 2]]) -> kỳ vọng [1296, 220296870]


## 3. Kiến thức nền

### 3.1. Phần tử lớn thứ `k` trên danh sách tăng dần

Nếu `a` được **sắp xếp tăng dần**, phần tử lớn nhất là `a[-1]`, lớn thứ hai là `a[-2]`, ..., lớn thứ `k` là `a[-k]` (chỉ số âm đếm từ cuối). Với trùng lặp, mỗi bản sao có hạng riêng (như Ví dụ 2: hai số 7 là hạng 1 và 2).

In [2]:
a = sorted([7, 5, 4, 7])
print(a)                      # [4, 5, 7, 7]
for k in range(1, 5):
    print(f"lớn thứ {k}: a[-{k}] = {a[-k]}")

[4, 5, 7, 7]
lớn thứ 1: a[-1] = 7
lớn thứ 2: a[-2] = 7
lớn thứ 3: a[-3] = 5
lớn thứ 4: a[-4] = 4


### 3.2. `bisect.insort` — chèn giữ thứ tự

`bisect.insort(a, x)` tìm vị trí chèn bằng tìm nhị phân (`O(log n)`) rồi chèn (`list.insert`, dịch các phần tử phía sau: `O(n)` nhưng là `memmove` rất nhanh của C).

In [3]:
a = [2, 3, 5, 8]
bisect.insort(a, 4)
bisect.insort(a, 5)           # trùng: chèn cạnh bản sao
print(a)
print(bisect.bisect_left(a, 5), bisect.bisect_right(a, 5))   # 3 5: khoảng vị trí của số 5

[2, 3, 4, 5, 5, 8]
3 5


### 3.3. `pow(base, exp, mod)` — lũy thừa modulo

`pow(p, x, MOD)` tính `p^x mod MOD` bằng **bình phương liên tiếp** trong `O(log x)` phép nhân, mỗi phép nhân giữ số trong `[0, MOD)`. Không bao giờ tính `p**x` rồi mới `%` (số có hàng tỷ chữ số).

Nguyên tắc: `x = Σ bit_i · 2^i` nên `p^x = Π_{bit_i = 1} p^{2^i}`; ta bình phương `p` dần dần và nhân vào kết quả khi bit tương ứng là 1.

In [4]:
def mod_pow(base, exp, mod):
    result = 1
    base %= mod
    while exp > 0:
        if exp & 1:
            result = result * base % mod
        base = base * base % mod
        exp >>= 1
    return result

print(mod_pow(6, 4, MOD), mod_pow(1296, 7, MOD))
assert all(mod_pow(b, e, MOD) == pow(b, e, MOD) for b in (2, 6, 10**9) for e in (0, 1, 2, 10**9))
import math
print("số chữ số của 2**100000 ≈", int(100000 * math.log10(2)) + 1, " (vs. pow ba tham số luôn < 1e9+7)")

1296 220296870
số chữ số của 2**100000 ≈ 30103  (vs. pow ba tham số luôn < 1e9+7)


### 3.4. Cây Fenwick (BIT) — đếm tần suất theo giá trị

Cây Fenwick lưu **tổng tiền tố** của một mảng đếm với hai thao tác `O(log n)`:

- `add(i, 1)`: tăng đếm tại vị trí `i`;
- `prefix(i)`: tổng đếm các vị trí `1..i`.

Với dữ liệu là **giá trị** ta dùng "nén tọa độ": đánh số các giá trị khác nhau theo thứ tự tăng dần (hạng `1..m`), vị trí `i` đếm số lần xuất hiện của giá trị hạng `i`.

In [5]:
class Fenwick:
    def __init__(self, n):
        self.n = n
        self.t = [0] * (n + 1)
    def add(self, i, v=1):
        while i <= self.n:
            self.t[i] += v
            i += i & -i
    def prefix(self, i):
        s = 0
        while i > 0:
            s += self.t[i]
            i -= i & -i
        return s

values = [7, 5, 4, 7]
comp = {v: i + 1 for i, v in enumerate(sorted(set(values)))}      # 4->1, 5->2, 7->3
print(comp)
fw = Fenwick(len(comp))
for v in values:
    fw.add(comp[v])
print([fw.prefix(i) for i in range(1, 4)], "<- số phần tử ≤ 4, ≤ 5, ≤ 7")

{4: 1, 5: 2, 7: 3}
[1, 2, 4] <- số phần tử ≤ 4, ≤ 5, ≤ 7


### 3.5. Tìm phần tử nhỏ thứ `k` bằng bước nhảy nhị phân trên Fenwick

Muốn tìm hạng (vị trí nén) `pos` nhỏ nhất sao cho `prefix(pos) ≥ k`: bắt đầu `pos = 0`, thử các bước `2^j` từ lớn đến nhỏ; nếu `t[pos + 2^j] < k` thì nhảy (`pos += 2^j`, `k -= t[pos]`). Cuối cùng đáp án là `pos + 1`. Mỗi `t[...]` là tổng của đúng đoạn bước đó nên cách này là tìm nhị phân trên tổng tiền tố trong `O(log n)`.

In [6]:
def fenwick_kth_smallest(fw, k):
    pos = 0
    step = 1
    while step * 2 <= fw.n:
        step *= 2
    while step:
        nxt = pos + step
        if nxt <= fw.n and fw.t[nxt] < k:
            pos = nxt
            k -= fw.t[nxt]
        step //= 2
    return pos + 1

inv = {r: v for v, r in comp.items()}
sorted_values = sorted(values)
for k in range(1, 5):
    r = fenwick_kth_smallest(fw, k)
    print(f"nhỏ thứ {k}: hạng {r} -> giá trị {inv[r]}")
    assert inv[r] == sorted_values[k - 1]

nhỏ thứ 1: hạng 1 -> giá trị 4
nhỏ thứ 2: hạng 2 -> giá trị 5
nhỏ thứ 3: hạng 3 -> giá trị 7
nhỏ thứ 4: hạng 3 -> giá trị 7


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 2**:

| Truy vấn | `nums` sau chèn (sắp tăng) | `k` | lớn thứ `k` | `p` |
|---|---|---|---|---|
| `[4,3]` | `[4,5,7]` | 3 | `a[-3] = 4` | `6^4 = 1296` |
| `[7,2]` | `[4,5,7,7]` | 2 | `a[-2] = 7` | `1296^7 mod M = 220296870` |

Việc cần làm mỗi truy vấn: (1) chèn một số vào tập có thứ tự; (2) đọc phần tử có hạng `k` từ phải; (3) một phép `pow`. Nếu giữ **mảng đã sắp xếp** thì (2) là `O(1)`; nếu phải sắp xếp lại mỗi lần thì (1)+(2) tốn `O(n log n)`.

In [7]:
nums, p = [7, 5], 6
a = sorted(nums)
out = []
for val, k in [[4, 3], [7, 2]]:
    bisect.insort(a, val)
    x = a[-k]
    p = pow(p, x, MOD)
    print(f"chèn {val}: a = {a}, k = {k}, x = {x}, p = {p}")
    out.append(p)
assert out == [1296, 220296870]

chèn 4: a = [4, 5, 7], k = 3, x = 4, p = 1296
chèn 7: a = [4, 5, 7, 7], k = 2, x = 7, p = 220296870


## 5. Cách 1 — Brute Force: sắp xếp lại sau mỗi lần chèn

**Ý tưởng.** Giữ `nums` thường, `append(val)` rồi `sorted(nums, reverse=True)[k−1]`.

In [8]:
def brute(nums, p, queries):
    cur = list(nums)
    out = []
    for val, k in queries:
        cur.append(val)
        x = sorted(cur, reverse=True)[k - 1]
        p = pow(p, x, MOD)
        out.append(p)
    return out

for args, exp in EXAMPLES:
    got = brute(*args)
    print(got)
    assert got == exp

[64, 4096]
[1296, 220296870]


**Độ phức tạp.** Mỗi truy vấn sắp xếp `O((n + m) log(n + m))` ⇒ tổng `O(m · (n + m) log(n + m))`: với `n = m = 2·10^4` là cỡ `2·10^4 · 4·10^4 · 15 = 1.2·10^10` — quá chậm; không gian `O(n + m)`.

**Nút thắt.** Sắp xếp lại toàn bộ để chỉ thêm **một** phần tử. Nếu giữ danh sách đã sắp xếp thì chỉ cần chèn đúng chỗ.

## 6. Bước đột phá

**Giữ tập số luôn có thứ tự.** Mỗi truy vấn chỉ thêm một phần tử, nên ta không cần sắp xếp lại: chèn vào đúng vị trí. Hai cấu trúc phù hợp:

- **Danh sách Python + `bisect.insort`:** tìm vị trí `O(log n)`, chèn `O(n)` nhưng `memmove` cực nhanh (`n ≤ 4·10^4` ⇒ rất nhẹ). Phần tử lớn thứ `k` là `a[-k]`.
- **Cây Fenwick trên giá trị nén:** vì **mọi** giá trị có thể xuất hiện (từ `nums` và các `val`) đều biết trước (ngoại tuyến), ta nén tọa độ một lần; chèn = `add(rank(val), 1)`; phần tử lớn thứ `k` = phần tử nhỏ thứ `size − k + 1` — tìm bằng bước nhảy nhị phân `O(log n)`.

**Vì sao heap không dùng được?** Heap chỉ cho phần tử nhỏ/lớn nhất. Nếu `k` cố định ta có thể giữ heap kích thước `k`; nhưng `k` đổi tùy ý giữa các truy vấn nên hai-heap phải chuyển nhiều phần tử mỗi lần — không bị chặn `O(log n)`.

**Cập nhật `p`:** luôn dùng `p` **hiện tại** làm cơ số: `p ← pow(p, x, MOD)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Danh sách đã sắp xếp + `bisect.insort`

**Giải thích từng đoạn.**

1. `a = sorted(nums)`.
2. Mỗi truy vấn: `bisect.insort(a, val)`; `x = a[-k]`; `p = pow(p, x, MOD)`; thêm `p` vào kết quả.

In [9]:
def with_insort(nums, p, queries):
    a = sorted(nums)
    out = []
    for val, k in queries:
        bisect.insort(a, val)
        p = pow(p, a[-k], MOD)
        out.append(p)
    return out

for args, exp in EXAMPLES:
    got = with_insort(*args)
    print(got)
    assert got == exp

[64, 4096]
[1296, 220296870]


**Độ phức tạp:** mỗi truy vấn `O(log n)` tìm vị trí + `O(n)` dịch (memmove) + `O(log x)` lũy thừa ⇒ tổng `O((n + m)·m)` phép dịch cực nhanh trong C (cỡ `10^9` byte-chuyển — chạy trong khoảng một phần nhỏ giây ở giới hạn đề). Không gian `O(n + m)`.

### 7.2. Cách 3 — Cây Fenwick trên giá trị nén

**Giải thích từng đoạn.**

1. Gom mọi giá trị `nums` và `val` của truy vấn, sắp xếp khác nhau, gán hạng `1..m`.
2. Khởi tạo Fenwick với `nums`.
3. Truy vấn: `add(rank[val], 1)`, `size += 1`, tìm hạng nhỏ thứ `size − k + 1` (chính là lớn thứ `k`), đổi hạng về giá trị, `p = pow(p, x, MOD)`.

In [10]:
def with_fenwick(nums, p, queries):
    allv = sorted(set(nums) | {v for v, _ in queries})
    rank = {v: i + 1 for i, v in enumerate(allv)}
    fw = Fenwick(len(allv))
    for v in nums:
        fw.add(rank[v])
    size = len(nums)
    out = []
    for val, k in queries:
        fw.add(rank[val])
        size += 1
        r = fenwick_kth_smallest(fw, size - k + 1)       # lớn thứ k = nhỏ thứ size - k + 1
        p = pow(p, allv[r - 1], MOD)
        out.append(p)
    return out

for args, exp in EXAMPLES:
    got = with_fenwick(*args)
    print(got)
    assert got == exp

[64, 4096]
[1296, 220296870]


**Độ phức tạp:** `O((n + m) log(n + m))` để nén và dựng; mỗi truy vấn `O(log(n + m)) + O(log x)`. Tổng `O((n + m) log(n + m) + m log x)`. Không gian `O(n + m)`. Cách này có **bảo đảm tiệm cận** thật sự, không phụ thuộc `memmove`.

### 7.3. Cách 4 — Danh sách có thứ tự theo khối (mini `SortedList`)

**Ý tưởng.** Thư viện `sortedcontainers.SortedList` (lời giải doocs dùng) lưu các số trong nhiều **khối** nhỏ được sắp xếp; chèn chỉ dịch một khối (kích thước cỡ `√n`), truy cập theo hạng đi qua các khối. Ta tự cài phiên bản tối giản để không phụ thuộc thư viện (LeetCode có sẵn `sortedcontainers`, nhưng môi trường khác có thể không).

**Giải thích từng đoạn.**

1. `blocks`: danh sách các list con đã sắp xếp; `maxes[j]` là phần tử lớn nhất của khối `j`.
2. `add(x)`: tìm khối đầu tiên có `max ≥ x` bằng `bisect_left(maxes, x)` (nếu không có thì khối cuối); `insort` vào khối đó; nếu khối quá lớn (`> 2·LOAD`) thì tách đôi.
3. `kth_largest(k)`: duyệt các khối từ cuối; trừ kích thước khối cho tới khi `k` rơi vào một khối.

In [11]:
class MiniSortedList:
    LOAD = 200

    def __init__(self, items=()):
        items = sorted(items)
        self.blocks = [items[i:i + self.LOAD] for i in range(0, len(items), self.LOAD)] or [[]]
        self.maxes = [b[-1] if b else None for b in self.blocks]
        self.size = len(items)

    def add(self, x):
        if self.size == 0:
            self.blocks[0].append(x)
            self.maxes[0] = x
        else:
            j = bisect.bisect_left(self.maxes, x)
            if j == len(self.blocks):
                j -= 1
            bisect.insort(self.blocks[j], x)
            self.maxes[j] = self.blocks[j][-1]
            if len(self.blocks[j]) > 2 * self.LOAD:               # tách đôi khối quá lớn
                half = len(self.blocks[j]) // 2
                right = self.blocks[j][half:]
                del self.blocks[j][half:]
                self.blocks.insert(j + 1, right)
                self.maxes[j] = self.blocks[j][-1]
                self.maxes.insert(j + 1, right[-1])
        self.size += 1

    def kth_largest(self, k):
        for b in reversed(self.blocks):
            if k <= len(b):
                return b[-k]
            k -= len(b)
        raise IndexError("k vượt quá kích thước")

def with_blocks(nums, p, queries):
    sl = MiniSortedList(nums)
    out = []
    for val, k in queries:
        sl.add(val)
        p = pow(p, sl.kth_largest(k), MOD)
        out.append(p)
    return out

for args, exp in EXAMPLES:
    got = with_blocks(*args)
    print(got)
    assert got == exp

[64, 4096]
[1296, 220296870]


**Độ phức tạp:** thêm `O(log(n/B) + B)` với kích thước khối `B ≈ 200`; truy cập hạng `O(n/B)` (có thể giảm bằng cộng dồn kích thước khối). Với `n ≤ 4·10^4`: `n/B ≈ 200` bước đơn giản. `SortedList` thật dùng cây chỉ số để đạt `O(log n)` gần như. Không gian `O(n)`.

## 8. TRACE từng bước

Trace **Cách 3 (Fenwick)** trên Ví dụ 2: `nums = [7,5]`, `p = 6`, `queries = [[4,3],[7,2]]`.

Giá trị xuất hiện: `{4, 5, 7}` → hạng `4→1`, `5→2`, `7→3`. Fenwick khởi tạo với `7` và `5`: đếm theo hạng `[0, 1, 1]` (hạng 2 và 3 mỗi nơi 1).

| Bước | thêm | đếm theo hạng | `size` | `k` | cần nhỏ thứ | hạng tìm được → giá trị | `p` |
|---|---|---|---|---|---|---|---|
| 1 | 4 (hạng 1) | `[1,1,1]` | 3 | 3 | `3 − 3 + 1 = 1` | hạng 1 → `4` | `6^4 = 1296` |
| 2 | 7 (hạng 3) | `[1,1,2]` | 4 | 2 | `4 − 2 + 1 = 3` | hạng 3 → `7` | `1296^7 mod M` |

Chạy trace để thấy cả mảng `t` của Fenwick và từng bước nhảy nhị phân.

In [12]:
def trace_fenwick(nums, p, queries):
    allv = sorted(set(nums) | {v for v, _ in queries})
    rank = {v: i + 1 for i, v in enumerate(allv)}
    print("hạng:", rank)
    fw = Fenwick(len(allv))
    for v in nums:
        fw.add(rank[v])
    size = len(nums)
    print("khởi tạo: t =", fw.t[1:], "size =", size)
    out = []
    for qi, (val, k) in enumerate(queries):
        fw.add(rank[val]); size += 1
        target = size - k + 1
        print(f"\ntruy vấn {qi}: chèn {val} (hạng {rank[val]}), t = {fw.t[1:]}, size = {size}")
        print(f"  lớn thứ {k} = nhỏ thứ {target}")
        pos, kk, step = 0, target, 1
        while step * 2 <= fw.n:
            step *= 2
        while step:
            nxt = pos + step
            if nxt <= fw.n and fw.t[nxt] < kk:
                print(f"  bước nhảy {step}: t[{nxt}] = {fw.t[nxt]} < {kk} -> pos = {nxt}, k còn {kk - fw.t[nxt]}")
                pos = nxt; kk -= fw.t[nxt]
            else:
                print(f"  bước nhảy {step}: không nhảy (nxt = {nxt}" + (f", t[{nxt}] = {fw.t[nxt]} >= {kk})" if nxt <= fw.n else " vượt n)"))
            step //= 2
        x = allv[pos]                 # hạng pos + 1 -> chỉ số mảng pos
        p = pow(p, x, MOD)
        print(f"  hạng {pos + 1} -> giá trị {x}; p = {p}")
        out.append(p)
    return out

assert trace_fenwick([7, 5], 6, [[4, 3], [7, 2]]) == [1296, 220296870]

hạng: {4: 1, 5: 2, 7: 3}
khởi tạo: t = [0, 1, 1] size = 2

truy vấn 0: chèn 4 (hạng 1), t = [1, 2, 1], size = 3
  lớn thứ 3 = nhỏ thứ 1
  bước nhảy 2: không nhảy (nxt = 2, t[2] = 2 >= 1)
  bước nhảy 1: không nhảy (nxt = 1, t[1] = 1 >= 1)
  hạng 1 -> giá trị 4; p = 1296

truy vấn 1: chèn 7 (hạng 3), t = [1, 2, 2], size = 4
  lớn thứ 2 = nhỏ thứ 3
  bước nhảy 2: t[2] = 2 < 3 -> pos = 2, k còn 1
  bước nhảy 1: không nhảy (nxt = 3, t[3] = 2 >= 1)
  hạng 3 -> giá trị 7; p = 220296870


## 9. Code nộp LeetCode

Nộp bản **Fenwick** (không phụ thuộc thư viện ngoài, có bảo đảm `O(log n)`). Có thể thay bằng `SortedList` của `sortedcontainers` hoặc `bisect.insort` nếu muốn mã ngắn hơn.

In [13]:
class Solution:
    def powerUpdate(self, nums: List[int], p: int, queries: List[List[int]]) -> List[int]:
        MOD = 10**9 + 7
        allv = sorted(set(nums) | {v for v, _ in queries})
        rank = {v: i + 1 for i, v in enumerate(allv)}
        m = len(allv)
        tree = [0] * (m + 1)

        def add(i):
            while i <= m:
                tree[i] += 1
                i += i & -i

        top = 1
        while top * 2 <= m:
            top *= 2

        def kth_smallest(k):
            pos, step = 0, top
            while step:
                nxt = pos + step
                if nxt <= m and tree[nxt] < k:
                    pos = nxt
                    k -= tree[nxt]
                step >>= 1
            return pos            # chỉ số trong allv (hạng pos + 1)

        for v in nums:
            add(rank[v])
        size = len(nums)
        ans = []
        for val, k in queries:
            add(rank[val])
            size += 1
            x = allv[kth_smallest(size - k + 1)]
            p = pow(p, x, MOD)
            ans.append(p)
        return ans

sol = Solution()
for args, exp in EXAMPLES:
    got = sol.powerUpdate(*args)
    print(got)
    assert got == exp

[64, 4096]
[1296, 220296870]


## 10. Vì sao thuật toán đúng

**Phần tử lớn thứ `k`.** Với tập `S` (có lặp, kích thước `s`), phần tử lớn thứ `k` là phần tử ở vị trí `s − k + 1` (1-indexed) khi sắp tăng dần; tức nhỏ thứ `s − k + 1`. Các bản sao của cùng một giá trị chiếm các hạng liên tiếp — đúng như Ví dụ 2.

**Fenwick tìm hạng.** `fw.t[i]` lưu tổng đếm của đoạn `(i − lowbit(i), i]`. Bước nhảy nhị phân duy trì: `pos` là hạng lớn nhất sao cho tổng đếm các hạng `1..pos` `< k` (khi đã trừ phần đã đi qua vào `k`). Mỗi bước thử `pos + 2^j` với `j` giảm dần; điều kiện `t[pos + 2^j] < k` kiểm tra đúng tổng đoạn `(pos, pos + 2^j]` vì `pos` là bội của `2^{j+1}` tại thời điểm đó. Kết thúc, hạng `pos + 1` là hạng nhỏ nhất có `prefix ≥ k` — chính là giá trị nhỏ thứ `k`.

**Cập nhật `p`.** Đề định nghĩa `p ← p^x mod (10^9+7)` tuần tự; ta chỉ cần thực hiện đúng định nghĩa với `pow` ba tham số, vốn cho kết quả đúng của `p^x mod M`. `p` luôn nằm trong `[1, M−1]` (vì `M` nguyên tố và `p` không chia hết cho `M`) nên không có vấn đề `0^x`.

**Ngoại tuyến.** Mọi giá trị có thể chèn đã biết trước nên nén tọa độ hợp lệ; Fenwick kích thước cố định `m`.

## 11. Phân tích Time/Space

Đặt `n = len(nums)`, `m = len(queries)`, `T = n + m`.

| Bước | Thời gian |
|---|---|
| Sắp xếp + nén tọa độ | `O(T log T)` |
| Dựng Fenwick từ `nums` | `O(n log T)` |
| Mỗi truy vấn: `add` + tìm hạng | `O(log T)` |
| Mỗi truy vấn: `pow(p, x, MOD)` | `O(log x) ≈ 30` phép nhân modulo |

**Tổng:** `O(T log T + m log x)` thời gian, `O(T)` không gian. Với `T = 4·10^4`: vài chục nghìn phép Fenwick — chạy dưới một giây.

So sánh: `insort` là `O(m · T)` thao tác dịch bộ nhớ (nhanh trong thực tế); brute force `O(m · T log T)`.

## 12. Bộ test

Ví dụ đề, edge case (`k = 1`, `k = kích thước`, nhiều số trùng, giá trị biên `10^9`), phản ví dụ, và đối chiếu ngẫu nhiên giữa 5 cách có seed.

In [14]:
sol = Solution()
for args, exp in EXAMPLES:
    assert sol.powerUpdate(*args) == exp

assert sol.powerUpdate([5], 3, [[5, 1]]) == [pow(3, 5, MOD)]                  # trùng giá trị
assert sol.powerUpdate([5], 3, [[1, 2]]) == [3]                               # k = kích thước -> nhỏ nhất = 1 -> p^1
assert sol.powerUpdate([10**9], 10**9, [[10**9, 1]]) == [pow(10**9, 10**9, MOD)]   # giá trị biên
assert sol.powerUpdate([1], 7, [[1, 1], [1, 2], [1, 3]]) == [7, 7, 7]         # toàn số 1: p^1
print("ví dụ + edge case OK")

ví dụ + edge case OK


In [15]:
def gen_case(rng, nmax, mmax, vmax):
    n = rng.randint(1, nmax)
    m = rng.randint(1, mmax)
    nums = [rng.randint(1, vmax) for _ in range(n)]
    p = rng.randint(1, 10**9)
    queries = [[rng.randint(1, vmax), rng.randint(1, n + i + 1)] for i in range(m)]
    return nums, p, queries

rng = random.Random(3930)
for _ in range(300):
    args = gen_case(rng, 12, 15, rng.choice([3, 10, 10**9]))
    exp = brute(*args)
    assert with_insort(*args) == exp, args
    assert with_fenwick(*args) == exp, args
    assert with_blocks(*args) == exp, args
    assert sol.powerUpdate(*args) == exp, args
print("300 test nhỏ: 5 cách khớp nhau")

# MiniSortedList với nhiều lần tách khối: đặt LOAD nhỏ để kích hoạt tách
MiniSortedList.LOAD = 3
for _ in range(100):
    args = gen_case(rng, 30, 60, rng.choice([5, 50]))
    assert with_blocks(*args) == with_insort(*args), args
MiniSortedList.LOAD = 200
print("MiniSortedList đúng cả khi tách khối thường xuyên")

300 test nhỏ: 5 cách khớp nhau
MiniSortedList đúng cả khi tách khối thường xuyên


In [16]:
# Thời gian ở giới hạn: n = m = 2e4
rng = random.Random(1)
n = m = 20000
nums = [rng.randint(1, 10**9) for _ in range(n)]
p0 = rng.randint(1, 10**9)
queries = [[rng.randint(1, 10**9), rng.randint(1, n + i + 1)] for i in range(m)]
results = {}
for name, fn in [("insort", with_insort), ("fenwick (Solution)", sol.powerUpdate), ("khối", with_blocks)]:
    t0 = time.time(); results[name] = fn(nums, p0, queries); print(f"{name:20s} {time.time() - t0:.2f}s")
assert results["insort"] == results["fenwick (Solution)"] == results["khối"]
t0 = time.time(); small_q = [[v, min(k, 2000 + i + 1)] for i, (v, k) in enumerate(queries[:300])]; r = brute(nums[:2000], p0, small_q); print(f"brute (n=2000, m=300) {time.time() - t0:.2f}s — tăng rất nhanh theo kích thước")

insort               0.10s


fenwick (Solution)   0.13s


khối                 0.11s
brute (n=2000, m=300) 0.08s — tăng rất nhanh theo kích thước


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Nhầm lớn thứ `k` với nhỏ thứ `k`

**Code sai:** `a[k-1]` trên danh sách tăng dần.
**Vì sao sai:** `a[k-1]` là nhỏ thứ `k`.
**Code đúng:** `a[-k]`.

In [17]:
def wrong_kth_smallest(nums, p, queries):
    a = sorted(nums); out = []
    for val, k in queries:
        bisect.insort(a, val)
        p = pow(p, a[k - 1], MOD)           # SAI: nhỏ thứ k
        out.append(p)
    return out

print("sai :", wrong_kth_smallest([2], 4, [[3, 1], [1, 2]]), " đúng:", Solution().powerUpdate([2], 4, [[3, 1], [1, 2]]))

sai : [16, 256]  đúng: [64, 4096]


### Lỗi 2 — Hỏi trước khi chèn

**Code sai:** lấy `x` rồi mới chèn `val`.
**Vì sao sai:** đề yêu cầu chèn **trước** khi tìm phần tử lớn thứ `k` (ràng buộc `k ≤ n + i + 1` đã gồm phần tử mới).
**Code đúng:** `insort` rồi `a[-k]`.

In [18]:
def wrong_query_first(nums, p, queries):
    a = sorted(nums); out = []
    for val, k in queries:
        x = a[-k] if k <= len(a) else a[0]      # SAI: hỏi trước khi chèn
        bisect.insort(a, val)
        p = pow(p, x, MOD)
        out.append(p)
    return out

print("sai :", wrong_query_first([2], 4, [[3, 1], [1, 2]]), " đúng:", Solution().powerUpdate([2], 4, [[3, 1], [1, 2]]))

sai : [16, 256]  đúng: [64, 4096]


### Lỗi 3 — Luôn dùng `p` ban đầu làm cơ số

**Code sai:** `ans.append(pow(p0, x, MOD))`.
**Vì sao sai:** `p` được **cập nhật** sau mỗi truy vấn; truy vấn sau dùng `p` mới.
**Code đúng:** `p = pow(p, x, MOD)`.

In [19]:
def wrong_original_p(nums, p0, queries):
    a = sorted(nums); out = []
    for val, k in queries:
        bisect.insort(a, val)
        out.append(pow(p0, a[-k], MOD))        # SAI: dùng p0
    return out

print("sai :", wrong_original_p([2], 4, [[3, 1], [1, 2]]), " đúng:", Solution().powerUpdate([2], 4, [[3, 1], [1, 2]]))

sai : [64, 16]  đúng: [64, 4096]


### Lỗi 4 — Tính `p ** x` rồi mới `% MOD`

`p ** x` với `x ~ 10^9` có hàng tỷ chữ số — hết bộ nhớ/thời gian. Luôn `pow(p, x, MOD)`.

In [20]:
import time as _t
t0 = _t.time(); big = 7 ** 300000; t1 = _t.time() - t0
t0 = _t.time(); small = pow(7, 300000, MOD); t2 = _t.time() - t0
print(f"7**300000: ≈ {int(big.bit_length() * 0.30103) + 1} chữ số, {t1:.3f}s  |  pow(7, 300000, MOD): {small}, {t2:.6f}s  (x = 1e9 thì cách đầu không chạy nổi)")
assert big % MOD == small

7**300000: ≈ 253530 chữ số, 0.022s  |  pow(7, 300000, MOD): 347822114, 0.000039s  (x = 1e9 thì cách đầu không chạy nổi)


### Lỗi 5 — Sắp xếp lại toàn bộ mỗi truy vấn (TLE) hoặc lệch hạng khi nén tọa độ

- Sort lại mỗi lần: `O(m · T log T)` (xem Cách 1).
- Nén tọa độ **chỉ từ `nums`** mà quên các `val` của truy vấn → `KeyError`. Phải nén trên **hợp** `nums ∪ {val}`.
- Fenwick đánh số từ `1`, không từ `0`: đánh `0` sẽ lặp vô hạn (`i += i & -i` với `i = 0`).

## 14. Mẹo & nhận dạng pattern

- **"Chèn dần + hỏi phần tử thứ k (k bất kỳ)"** → danh sách có thứ tự (`SortedList`/`insort`) hoặc **Fenwick/segment tree trên giá trị nén** với tìm hạng bằng bước nhảy nhị phân.
- Khi `k` **cố định**, hãy dùng heap kích thước `k`; khi `k` thay đổi → cấu trúc theo hạng.
- **Truy vấn có thể đọc trước (ngoại tuyến)** → nén tọa độ trên mọi giá trị sẽ xuất hiện rồi dùng cấu trúc kích thước cố định.
- Lớn thứ `k` ⇔ nhỏ thứ `size − k + 1`: đổi chiều một lần để dùng hàm "nhỏ thứ k".
- Lũy thừa modulo luôn dùng `pow(a, b, MOD)`; không bao giờ tính `a**b` rồi mới modulo.
- `bisect.insort` rất nhanh khi `n ≤ 10^5` vì `memmove`; vẫn nên biết giải pháp `O(log n)` thật sự.

## 15. Biến thể

1. **Xóa phần tử:** Fenwick `add(rank, −1)`; `SortedList.remove`; `insort` + `list.pop(i)`.
2. **Trung vị động:** `k = (size + 1) // 2` — hai heap hoặc cấu trúc trên.
3. **Tổng của `k` phần tử lớn nhất:** Fenwick thứ hai lưu tổng giá trị; hoặc segment tree.
4. **Đếm số phần tử lớn hơn `v`:** `size − prefix(rank[v])`.
5. **Các truy vấn trực tuyến (không biết trước `val`):** dùng `SortedList`/cây cân bằng hoặc segment tree động.

In [21]:
# Biến thể 2 + 4: trung vị động và đếm số phần tử > v bằng Fenwick
def dynamic_median_and_greater(nums, queries):
    allv = sorted(set(nums) | {v for v, _ in queries})
    rank = {v: i + 1 for i, v in enumerate(allv)}
    fw = Fenwick(len(allv))
    for v in nums:
        fw.add(rank[v])
    size = len(nums)
    out = []
    for val, _ in queries:
        fw.add(rank[val]); size += 1
        median = allv[fenwick_kth_smallest(fw, (size + 1) // 2) - 1]
        greater = size - fw.prefix(rank[val])
        out.append((median, greater))
    return out

res = dynamic_median_and_greater([5, 1, 9], [[4, 0], [10, 0], [1, 0]])
a = [5, 1, 9]
for (val, _), (med, gr) in zip([[4, 0], [10, 0], [1, 0]], res):
    a.append(val)
    s = sorted(a)
    assert med == s[(len(s) + 1) // 2 - 1] and gr == sum(1 for x in a if x > val)
print(res)

[(4, 2), (5, 0), (4, 4)]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sort lại mỗi lần | `sorted(cur)[...]` mỗi truy vấn | `O(m·T log T)` | `O(T)` | kiểm chứng, `T ≲ 10^3` |
| 2. `bisect.insort` | giữ danh sách tăng, `a[-k]` | `O(m·T)` memmove | `O(T)` | **nhanh, mã ngắn**; `T ≤ 10^5` |
| 3. Fenwick trên giá trị nén | đếm + bước nhảy nhị phân | `O(T log T)` | `O(T)` | **lời giải nộp**; bảo đảm tiệm cận |
| 4. Danh sách khối (mini `SortedList`) | các khối `~200` phần tử, truy cập theo khối | `O(m·(B + T/B))` | `O(T)` | trực tuyến, không cần biết trước giá trị |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** Với `a = [1, 3, 3, 8]` (đã sắp tăng), `a[-2]` bằng bao nhiêu? Đó là phần tử lớn thứ mấy?

**Bài 2 (tính tay).** `nums = [3]`, `p = 2`, `queries = [[5,1],[1,3]]`. Hai giá trị `p` là gì? (`2^5 = 32`, rồi lớn thứ 3 của `[3,5,1]`.)

**Bài 3 (điền TODO).** Hoàn thiện `my_kth_largest(fw, size, k)` dùng `fenwick_kth_smallest`.

**Bài 4 (tự viết lại).** Viết `my_power_update(nums, p, queries)` bằng `bisect.insort` (Cách 2), không nhìn code.

**Bài 5 (biến thể).** Viết `sum_of_top_k(nums, queries)`: sau mỗi chèn `[val, k]` trả về **tổng** `k` phần tử lớn nhất (dùng danh sách sắp xếp là đủ).

In [22]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_kth_largest(fw, size, k):
    # TODO: trả về HẠNG (vị trí nén) của phần tử lớn thứ k: hạng nhỏ thứ (size - k + 1)
    pass

# Bài 4 — TODO
def my_power_update(nums, p, queries):
    # TODO: a = sorted(nums); với mỗi (val, k): insort, x = a[-k], p = pow(p, x, MOD)
    pass

# Bài 5 — TODO
def sum_of_top_k(nums, queries):
    # TODO: giữ danh sách tăng; sau chèn: sum(a[-k:])
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: đếm từ cuối: `a[-1] = 8`, `a[-2] = 3`.
- Mức 2: hai số 3 chiếm hạng 2 và 3 từ trên.
- Mức 3: `a[-2] = 3`, lớn thứ hai.

**Bài 2.**
- Mức 1: chèn 5 → `[3,5]`, lớn thứ 1 là `5`, `p = 32`.
- Mức 2: chèn 1 → `[1,3,5]`, lớn thứ 3 là `1`.
- Mức 3: `p = 32^1 = 32`; kết quả `[32, 32]`.

**Bài 3.**
- Mức 1: lớn thứ `k` ⇔ nhỏ thứ `size − k + 1`.
- Mức 2: gọi `fenwick_kth_smallest(fw, size - k + 1)`.
- Mức 3: hàm trả về hạng, không phải giá trị.

**Bài 4.**
- Mức 1: xem lại `with_insort`.
- Mức 2: `p` được cập nhật liên tục.
- Mức 3: `a[-k]` sau khi `insort`.

**Bài 5.**
- Mức 1: `a[-k:]` là `k` phần tử lớn nhất.
- Mức 2: chèn rồi mới tính tổng.
- Mức 3: `sum(a[-k:])`.

## 19. Đáp án tham khảo

In [23]:
# Bài 1
a = [1, 3, 3, 8]
assert a[-2] == 3
# Bài 2
assert Solution().powerUpdate([3], 2, [[5, 1], [1, 3]]) == [32, 32] == brute([3], 2, [[5, 1], [1, 3]])

# Bài 3
def my_kth_largest_ans(fw, size, k):
    return fenwick_kth_smallest(fw, size - k + 1)

vals = [4, 8, 8, 2, 9]
comp2 = {v: i + 1 for i, v in enumerate(sorted(set(vals)))}
inv2 = {r: v for v, r in comp2.items()}
fw2 = Fenwick(len(comp2))
for v in vals:
    fw2.add(comp2[v])
sv = sorted(vals)
for k in range(1, 6):
    assert inv2[my_kth_largest_ans(fw2, 5, k)] == sv[-k]
print("Bài 1–3 OK")

Bài 1–3 OK


In [24]:
# Bài 4
def my_power_update_ans(nums, p, queries):
    a = sorted(nums); out = []
    for val, k in queries:
        bisect.insort(a, val)
        p = pow(p, a[-k], MOD)
        out.append(p)
    return out

rng = random.Random(30)
for _ in range(100):
    args = gen_case(rng, 10, 12, 20)
    assert my_power_update_ans(*args) == brute(*args)

# Bài 5
def sum_of_top_k_ans(nums, queries):
    a = sorted(nums); out = []
    for val, k in queries:
        bisect.insort(a, val)
        out.append(sum(a[-k:]))
    return out

assert sum_of_top_k_ans([5, 1], [[3, 2], [10, 1], [2, 4]]) == [8, 10, 20]
for _ in range(100):
    nums, _, qs = gen_case(rng, 10, 12, 20)
    exp = []
    cur = list(nums)
    for v, k in qs:
        cur.append(v)
        exp.append(sum(sorted(cur, reverse=True)[:k]))
    assert sum_of_top_k_ans(nums, qs) == exp
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (Fenwick trên giá trị nén, tìm hạng):**

```python
allv = sorted(set(nums) | {v for v, _ in queries}); rank = {v: i+1}
top = lũy thừa 2 lớn nhất <= m
def kth_smallest(k):
    pos, step = 0, top
    while step:
        if pos+step <= m and tree[pos+step] < k: pos += step; k -= tree[pos]
        step >>= 1
    return pos            # chỉ số trong allv
for val, k in queries:
    add(rank[val]); size += 1
    x = allv[kth_smallest(size - k + 1)]       # lớn thứ k = nhỏ thứ size-k+1
    p = pow(p, x, MOD)
```

**Template ngắn (Python):** `bisect.insort(a, val); p = pow(p, a[-k], MOD)`.

**Checklist trước khi submit:**

- [ ] Chèn **trước** khi hỏi; `k` tính trên tập đã có phần tử mới.
- [ ] Lớn thứ `k` = `a[-k]` (hoặc nhỏ thứ `size − k + 1`).
- [ ] `p` được cập nhật liên tục (`p = pow(p, x, MOD)`), không dùng `p` gốc.
- [ ] Dùng `pow` ba tham số, không `**`.
- [ ] Nén tọa độ trên `nums ∪ {val}`; Fenwick đánh số từ `1`.
- [ ] Các số trùng có hạng riêng (kiểm tra Ví dụ 2).
- [ ] Đã đối chiếu brute force với `n, m ≤ 15` và giá trị nhỏ (nhiều trùng).
- [ ] Đã đối chiếu chữ ký phương thức Premium trên LeetCode.